# PixiJoy Inpaint (M1) — Colab quality train

**Preferred:** Runtime → **GPU** → Run all.

**If GPU quota is blocked:** Runtime → Change runtime type → Hardware accelerator **None** + **High-RAM** → Save → Run all.

- CPU is **much slower** (hours–days vs GPU minutes–hours) but still **quality** (same data + gates). Drive resume keeps progress across disconnects.
- Checkpoints + data: `MyDrive/pixijoy_ml/inpaint/`
- Pin TFLite only if verify prints **OK** / **SHIP_OK**.


In [ ]:
!nvidia-smi || true
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TF", tf.__version__, "GPUs", gpus)
ALLOW_CPU = True  # set False to require GPU
if gpus:
    print('accelerator=GPU')
elif ALLOW_CPU:
    print('accelerator=CPU — use Runtime → High-RAM if available; expect slow train')
else:
    raise SystemExit('No GPU — select GPU runtime, or set ALLOW_CPU=True')


In [ ]:
USE_DRIVE = True  # persist ckpt + data on Drive (survive disconnect)

import os, urllib.request, zipfile, shutil
from pathlib import Path
from google.colab import files

ROOT = Path('/content/pixijoy_inpaint')
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/inpaint')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    (DRIVE / 'dist').mkdir(parents=True, exist_ok=True)
    (DRIVE / 'data').mkdir(parents=True, exist_ok=True)

def _link_to_drive(name: str) -> None:
    """Point local name/ at Drive so writes survive runtime death."""
    if not USE_DRIVE:
        Path(name).mkdir(parents=True, exist_ok=True)
        return
    local = Path(name)
    target = DRIVE / name
    target.mkdir(parents=True, exist_ok=True)
    if local.exists() or local.is_symlink():
        if local.is_symlink():
            if local.resolve() == target.resolve():
                print(f'{name} -> Drive OK')
                return
            local.unlink()
        elif local.is_dir():
            for p in local.rglob('*'):
                if not p.is_file():
                    continue
                dest = target / p.relative_to(local)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.is_file():
                    shutil.copy2(p, dest)
            shutil.rmtree(local)
        else:
            local.unlink()
    local.symlink_to(target)
    print(f'{name} -> {target}')

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/inpaint_lab.zip'
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, 'inpaint_lab.zip')
    with zipfile.ZipFile('inpaint_lab.zip') as z:
        z.extractall('.')
    ok = Path('train.py').is_file() and Path('prepare_quality_data.py').is_file()
    print('lab zip OK', ok)
except Exception as e:
    print('zip fetch failed:', e)
if not ok:
    print('Upload inpaint_lab.zip')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
    ok = Path('train.py').is_file()
assert ok, 'lab zip missing train.py / prepare_quality_data.py — refresh Runtime and re-fetch'
_link_to_drive('dist')
_link_to_drive('data')


In [ ]:
from pathlib import Path
print('prepare_quality_data (real data; skips if data/READY already on Drive)…')
rc = get_ipython().system('python -u prepare_quality_data.py --data ./data')
assert rc == 0, 'prepare_quality_data failed'
assert (Path('data') / 'READY').is_file(), 'data/READY missing — Drive link or prepare failed'
print('data READY ok')


In [ ]:
from pathlib import Path
import hashlib
import tensorflow as tf
USE_DRIVE = globals().get('USE_DRIVE', True)
OUT = Path('/content/drive/MyDrive/pixijoy_ml/inpaint/dist/inpaint.keras') if USE_DRIVE else Path('dist') / 'inpaint.keras'
OUT.parent.mkdir(parents=True, exist_ok=True)
ckpt = OUT.with_suffix('.ckpt.keras')
print('ckpt exists' if ckpt.is_file() else 'fresh train', OUT)
print('state', OUT.parent / 'train_state.json', (OUT.parent / 'train_state.json').is_file())

gpus = tf.config.list_physical_devices('GPU')
# CPU High-RAM: smaller batch to stay under memory; quality unchanged.
batch_flag = '' if gpus else ' --batch 4'
if not gpus:
    print('CPU mode: batch=4 (High-RAM recommended). Leave the tab open; resume after disconnect.')

cmd = "python -u train.py --quality --out \"" + str(OUT) + "\"" + batch_flag
if ckpt.is_file():
    cmd += " --resume"
    print('will resume from checkpoint')
print('RUN', cmd)
rc = get_ipython().system(cmd)
assert rc == 0, 'train failed'
rc = get_ipython().system('python -u export_tflite.py --keras "' + str(OUT) + '" --out "dist/inpaint.tflite"')
assert rc == 0, 'export failed (refusing untrained is correct)'
rc = get_ipython().system('python -u verify_contract.py --quality --data ./data')
assert rc == 0, 'verify --quality failed — do NOT pin'
p = Path('dist/inpaint.tflite')
assert p.is_file(), p
print('SHIP_OK', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download('dist/inpaint.tflite')
print('Attach inpaint.tflite to ml-packs-v1 ONLY if verify printed OK / SHIP_OK')
